# 05 — Build neural networks with nn.Module

**Level:** Beginner · **Study time:** 60–90 minutes (including experiments)

**Prerequisites:** Notebook 04; Python classes at a basic level.

## What you will be able to do
- Define layers and a forward method.
- Use a nonlinear activation.
- Compare a linear model with a multilayer perceptron (MLP).

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 04](04_linear_regression.ipynb) · [Next: 06](06_datasets_and_dataloaders.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Create a nonlinear prediction task

A straight line cannot describe a sine wave. Multiple linear layers alone are still a linear transformation; a nonlinear activation such as Tanh or ReLU changes the set of functions a model can represent.


In [ ]:
X = torch.linspace(-3, 3, 240).unsqueeze(1)
y = torch.sin(2 * X) + 0.06 * torch.randn_like(X)
train_idx = torch.arange(0, len(X), 2)
val_idx = torch.arange(1, len(X), 2)
plt.scatter(X[:, 0], y[:, 0], s=8); plt.show()


## Step 3 — Define a module

Assigning layers as attributes registers their parameters. `forward` describes how data flows through them. Calling `model(X)` runs forward through PyTorch's module machinery. `nn.Linear(in, out)` stores a weight shaped `[out, in]` and computes `X @ weight.T + bias`.


In [ ]:
class SineMLP(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                                    nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1))
    def forward(self, x):
        return self.layers(x)

model = SineMLP()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
assert model(X[:4]).shape == (4, 1)


## Step 4 — Train a linear baseline and an MLP

`MSELoss` replaces our hand-written squared error. Adam adapts update scales per parameter; it still depends on a useful learning rate. Both models see the same training observations.


In [ ]:
def fit(model, steps=500):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
    losses = []
    model.train()
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(X[train_idx]), y[train_idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

linear = nn.Linear(1, 1)
linear_history = fit(linear)
mlp_history = fit(model)
model.eval(); linear.eval()
with torch.no_grad():
    mlp_mse = nn.functional.mse_loss(model(X[val_idx]), y[val_idx]).item()
    linear_mse = nn.functional.mse_loss(linear(X[val_idx]), y[val_idx]).item()
print({"linear_validation_MSE": linear_mse, "MLP_validation_MSE": mlp_mse})
assert mlp_mse < linear_mse
plt.scatter(X[:, 0], y[:, 0], s=8, alpha=0.4)
plt.plot(X[:, 0], model(X).detach()[:, 0], label="MLP")
plt.plot(X[:, 0], linear(X).detach()[:, 0], label="linear")
plt.legend(); plt.show()


## Your turn — practice before opening the solutions

1. Count parameters in `nn.Linear(3, 5)` by hand, then check.
2. Build an MLP with ReLU and verify its output shape.
3. Compare your ReLU model on the same split. Keep the test set conceptually separate; this split is for development.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
layer = nn.Linear(3, 5)
assert sum(p.numel() for p in layer.parameters()) == 3 * 5 + 5
relu_model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
assert relu_model(X[:7]).shape == (7, 1)
_ = fit(relu_model)
relu_model.eval()
with torch.no_grad():
    print("ReLU validation MSE:", nn.functional.mse_loss(relu_model(X[val_idx]), y[val_idx]).item())


## Before moving on

Architecture controls which patterns the model can express; optimization determines which parameters it finds. A more complex model does not automatically generalize better.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Build a neural network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)


[Course guide](../README.md) · [Previous: 04](04_linear_regression.ipynb) · [Next: 06](06_datasets_and_dataloaders.ipynb)
